# Part II -- LLM SFT with Q-LoRA

**Assignment 4 -- Natural Language Processing**

Parameter-Efficient Fine-Tuning (PEFT) for **skill acquisition**: teaching a
decoder LLM to generate Python code via Supervised Fine-Tuning.

* **Base model:** `Qwen/Qwen2-1.5B-Instruct`
* **Quantization:** 4-bit NF4 (`BitsAndBytesConfig`) -- the frozen base weights
  live in 4-bit, only small LoRA adapters are trained (this is **Q-LoRA**).
* **Data:** a 5,000-example sample of `flytech/python-codes-25k` (Part II may use
  any sample above 2,000) wrapped with a system prompt into chat format.
* **Loss:** computed on the **assistant completion only** -- the system prompt
  and user query are masked out (`completion_only_loss`).
* **Output:** the trained LoRA adapter is merged into the base weights to produce
  the standalone SFT model that Part III aligns with DPO.

In [1]:
import os, json, time, gc

PROJECT_DIR = "/scratch/kotpaz/hamo-bassem/nlp"
os.environ.setdefault("HF_HOME", f"{PROJECT_DIR}/hf_cache")
for _k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE"):
    os.environ.setdefault(_k, "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, set_seed)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

set_seed(42)
SMOKE = os.environ.get("A4_SMOKE", "0") == "1"
print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("SMOKE mode:", SMOKE)

torch 2.6.0+cu124 | CUDA: True
GPU: NVIDIA H100 80GB HBM3
SMOKE mode: False


## 1. 4-bit quantization and model loading

`BitsAndBytesConfig` loads the 1.5B base model in 4-bit **NF4** with double
quantization; the compute dtype is bfloat16. `prepare_model_for_kbit_training`
makes the quantized model trainable (casts norms to fp32, enables input
gradients, wires up gradient checkpointing).

In [ ]:
MODEL_NAME  = "Qwen/Qwen2-1.5B-Instruct"
N_SAMPLES   = 64 if SMOKE else 5000
MAX_LEN     = 1024
ADAPTER_DIR = f"{PROJECT_DIR}/results/part2_sft/adapter"
MERGED_DIR  = f"{PROJECT_DIR}/results/part2_sft/merged"
TB_DIR      = f"{PROJECT_DIR}/tb_logs/part2_sft"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map={"": 0}, dtype=torch.bfloat16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
print(f"4-bit model footprint: {model.get_memory_footprint()/1e9:.2f} GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit model footprint: 1.59 GB


## 2. LoRA configuration

Rank `r=16`, applied to **all linear layers** (`target_modules="all-linear"`).
Only these low-rank adapter matrices are trained -- a tiny fraction of the 1.5B
parameters.

In [3]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## 3. Dataset -- chat formatting with a system prompt

Each example becomes a `prompt` / `completion` pair in chat form:
the system prompt + user request go in `prompt`, the reference code in
`completion`. TRL applies Qwen2's chat template; with `completion_only_loss=True`
the loss is back-propagated through the assistant tokens only.

In [4]:
SYSTEM_PROMPT = ("You are an expert Python programmer. Given a user request, "
                 "respond with correct, runnable Python code and a brief "
                 "explanation when helpful.")

raw = load_dataset("flytech/python-codes-25k", split="train")
raw = raw.shuffle(seed=42).select(range(N_SAMPLES))

def to_prompt_completion(ex):
    user = ex["instruction"].strip()
    if ex["input"] and ex["input"].strip():
        user += "\n\n" + ex["input"].strip()
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ],
        "completion": [
            {"role": "assistant", "content": ex["output"].strip()},
        ],
    }

sft_ds = raw.map(to_prompt_completion, remove_columns=raw.column_names)
print(sft_ds)
print("\n--- formatted example ---")
print("PROMPT    :", sft_ds[0]["prompt"])
print("COMPLETION:", sft_ds[0]["completion"])

Using the latest cached version of the dataset since flytech/python-codes-25k couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at /scratch/kotpaz/hamo-bassem/nlp/hf_cache/datasets/flytech___python-codes-25k/default/0.0.0/0ed98ff2a76c5d133d8c157b814189a5a17ebd20 (last modified on Thu May 21 22:36:00 2026).


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 5000
})

--- formatted example ---
PROMPT    : [{'role': 'system', 'content': 'You are an expert Python programmer. Given a user request, respond with correct, runnable Python code and a brief explanation when helpful.'}, {'role': 'user', 'content': 'Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order'}]
COMPLETION: [{'role': 'assistant', 'content': '```python\nimport random\n\n# generating a list of unique numbers from 0 to 9 in random order\nrandom_numbers = random.sample(range(0, 10), 10)\n\n# sort list of numbers \nrandom_numbers.sort()\n\n# print sorted list of random numbers\nprint(random_numbers)\n# Output: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]\n```'}]


## 4. Supervised fine-tuning (completion-only loss)

Hyper-parameters follow the assignment's SFT table: batch size 1 with gradient
accumulation 4 (effective batch 4), 1 epoch, LR 2e-4, `paged_adamw_8bit`,
max sequence length 1024. `SFTTrainer` receives the raw quantized model plus
`peft_config`, so it attaches the LoRA adapters itself.

In [5]:
sft_args = SFTConfig(
    output_dir=ADAPTER_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=8 if SMOKE else -1,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    max_length=MAX_LEN,
    completion_only_loss=True,
    packing=False,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=25,
    logging_dir=TB_DIR,
    save_strategy="no",
    report_to="tensorboard",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [6]:
t0 = time.time()
trainer.train()
print(f"\nSFT wall-time: {(time.time()-t0)/60:.1f} min")
trainer.save_model(ADAPTER_DIR)
print("LoRA adapter saved ->", ADAPTER_DIR)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
25,0.657438
50,0.616705
75,0.669542
100,0.665540
125,0.634501
150,0.592247
175,0.633124
200,0.529780
225,0.585961
250,0.559403



SFT wall-time: 25.2 min
LoRA adapter saved -> /scratch/kotpaz/hamo-bassem/nlp/results/part2_sft/adapter


/scratch/kotpaz/hamo-bassem/nlp/env/lib/python3.11/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen2-1.5B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


## 5. Merge the adapter

DPO in Part III needs a standalone reference model. We reload the base in
bfloat16 (un-quantized), attach the trained adapter and `merge_and_unload()` to
fold the LoRA weights into the base, then save the merged SFT model.

In [7]:
del model, trainer
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map={"": 0})
merged = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("merged SFT model saved ->", MERGED_DIR)

del base, merged
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

merged SFT model saved -> /scratch/kotpaz/hamo-bassem/nlp/results/part2_sft/merged


## 6. Qualitative check -- did it learn to code?

We generate from the merged SFT model. Unlike the Part I baseline it should now
produce coherent, runnable Python.

In [8]:
gen_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, dtype=torch.bfloat16, device_map={"": 0})
gen_model.eval()

def generate(user, system=SYSTEM_PROMPT, max_new_tokens=256):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)

TESTS = [
    "Write a Python function to compute the nth Fibonacci number.",
    "Write a Python program that counts word frequencies in a text file.",
]
for q in TESTS:
    print("USER:", q)
    print("SFT MODEL:\n" + generate(q))
    print("=" * 72)

del gen_model
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

USER: Write a Python function to compute the nth Fibonacci number.


SFT MODEL:
```python
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)
```
USER: Write a Python program that counts word frequencies in a text file.


SFT MODEL:
```python
import re

def count_words(filename):
    # Open the file for reading
    with open(filename) as f:
        # Read the contents of the file
        content = f.read()
    
    # Use regex to find all words
    words = re.findall(r'\w+', content)
    
    # Count the number of occurrences of each word
    word_count = {}
    for word in words:
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1
    
    return word_count
```


In [9]:
summary = {
    "part": "II - Q-LoRA SFT",
    "model": MODEL_NAME,
    "approach": "4-bit NF4 Q-LoRA, completion-only loss",
    "lora_rank": 16,
    "train_samples": N_SAMPLES,
}
with open(f"{PROJECT_DIR}/results/part2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "part": "II - Q-LoRA SFT",
  "model": "Qwen/Qwen2-1.5B-Instruct",
  "approach": "4-bit NF4 Q-LoRA, completion-only loss",
  "lora_rank": 16,
  "train_samples": 5000
}


## 7. Analysis

* **Efficiency.** Q-LoRA trains well under 1% of the parameters with the base
  weights frozen in 4-bit -- a fraction of the memory of Part I's full
  fine-tuning, yet it produces a far stronger generator.
* **Capability vs. control.** The SFT model is now *capable*: it writes
  functional Python. But it is purely compliant -- it will follow whatever the
  instruction asks. It has no notion of *when a request should be refused or
  corrected*. Supplying that behavioural control is exactly the job of Part III
  (DPO alignment), which builds directly on the merged model saved above.